# 01 — Dataset Analysis (EDA)

**Research**: Analisis Performa Algoritma Machine Learning untuk Klasifikasi Jenis Cyberbullying pada Teks Bahasa Indonesia Menggunakan TF-IDF

**Objective**: Perform Exploratory Data Analysis on the final dataset before preprocessing and model training.

**Important**: This notebook does NOT perform any preprocessing (no stemming, no stopword removal, no text cleaning). It analyzes the raw merged dataset as-is.

---

## Setup

In [ ]:
import sys
import os

# Ensure project root is in path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Import project modules
from src.config.settings import REPORTS_DIR, FIGURES_DIR
from src.data.load_dataset import load_dataset
from src.data.dataset_summary import (
    get_dataset_overview,
    get_missing_values,
    get_duplicate_count,
)
from src.data.text_analysis import (
    compute_text_lengths,
    get_text_statistics,
    get_vocabulary_stats,
    get_top_words,
    get_top_ngrams,
)
from src.data.label_analysis import (
    get_label_distribution,
    get_class_count,
    get_class_imbalance_ratio,
)
from src.visualization.plots import (
    plot_label_distribution,
    plot_label_distribution_pie,
    plot_text_length_histogram,
    plot_text_length_boxplot,
    plot_top_words,
    plot_top_ngrams,
    plot_missing_values,
    plot_duplicate_summary,
)
from src.utils.report_generator import (
    assess_data_quality,
    generate_eda_report,
    save_report,
    export_dataset_overview,
    export_label_distribution,
    export_text_statistics,
    export_vocabulary_statistics,
)

import matplotlib.pyplot as plt
%matplotlib inline

# Ensure output directories
os.makedirs(FIGURES_DIR, exist_ok=True)

print('Setup complete.')

---
## 1. Load Dataset

In [ ]:
df = load_dataset()
df.head(10)

---
## 2. Dataset Overview

In [ ]:
overview = get_dataset_overview(df)

print(f"Rows:    {overview['shape']['rows']:,}")
print(f"Columns: {overview['shape']['columns']}")
print(f"Memory:  {overview['memory_usage']['total_mb']} MB")
print(f"\nData Types:")
for col, dtype in overview['data_types'].items():
    print(f"  {col}: {dtype}")

### 2.1 Missing Values

In [ ]:
missing_df = get_missing_values(df)
display(missing_df)

fig = plot_missing_values(missing_df)
fig.savefig(os.path.join(FIGURES_DIR, 'missing_values.png'), bbox_inches='tight')
plt.show()

### 2.2 Duplicate Analysis

In [ ]:
duplicate_info = get_duplicate_count(df)
print(f"Duplicate Rows:       {duplicate_info['duplicate_count']:,}")
print(f"Duplicate Percentage: {duplicate_info['duplicate_percentage']}%")
print(f"Total Rows:           {duplicate_info['total_rows']:,}")

fig = plot_duplicate_summary(duplicate_info)
fig.savefig(os.path.join(FIGURES_DIR, 'duplicate_summary.png'), bbox_inches='tight')
plt.show()

---
## 3. Label Distribution

In [ ]:
label_dist = get_label_distribution(df)
class_count = get_class_count(df)

print(f"Number of Classes: {class_count}")
display(label_dist)

In [ ]:
fig = plot_label_distribution(label_dist)
fig.savefig(os.path.join(FIGURES_DIR, 'label_distribution.png'), bbox_inches='tight')
plt.show()

In [ ]:
fig = plot_label_distribution_pie(label_dist)
fig.savefig(os.path.join(FIGURES_DIR, 'label_distribution_pie.png'), bbox_inches='tight')
plt.show()

### 3.1 Class Imbalance Assessment

In [ ]:
imbalance = get_class_imbalance_ratio(df)

print(f"Imbalance Ratio: {imbalance['ratio']}:1")
print(f"Status:          {imbalance['status']}")
print(f"Largest Class:   {imbalance['max_class']} ({imbalance['max_count']:,})")
print(f"Smallest Class:  {imbalance['min_class']} ({imbalance['min_count']:,})")
print(f"\n{imbalance['explanation']}")

---
## 4. Text Length Analysis

In [ ]:
df_with_lengths = compute_text_lengths(df)
text_stats = get_text_statistics(df_with_lengths)

print('Word Count Statistics:')
for k, v in text_stats['word_count'].items():
    print(f'  {k:>8}: {v}')

print('\nCharacter Count Statistics:')
for k, v in text_stats['char_count'].items():
    print(f'  {k:>8}: {v}')

In [ ]:
fig = plot_text_length_histogram(df_with_lengths)
fig.savefig(os.path.join(FIGURES_DIR, 'text_length_histogram.png'), bbox_inches='tight')
plt.show()

In [ ]:
fig = plot_text_length_boxplot(df_with_lengths)
fig.savefig(os.path.join(FIGURES_DIR, 'text_length_boxplot.png'), bbox_inches='tight')
plt.show()

---
## 5. Word Frequency Analysis

In [ ]:
top_20_words = get_top_words(df, n=20)
top_50_words = get_top_words(df, n=50)

print('Top 20 Most Frequent Words:')
for word, count in top_20_words:
    print(f'  {word:>20}: {count:,}')

In [ ]:
fig = plot_top_words(top_20_words, title='Top 20 Most Frequent Words')
fig.savefig(os.path.join(FIGURES_DIR, 'top_words.png'), bbox_inches='tight')
plt.show()

---
## 6. Bigram Analysis

In [ ]:
top_bigrams = get_top_ngrams(df, n=2, top_k=20)

print('Top 20 Bigrams:')
for ngram, count in top_bigrams:
    print(f'  {ngram:>30}: {count:,}')

In [ ]:
fig = plot_top_ngrams(top_bigrams, title='Top 20 Bigrams')
fig.savefig(os.path.join(FIGURES_DIR, 'top_bigrams.png'), bbox_inches='tight')
plt.show()

---
## 7. Trigram Analysis

In [ ]:
top_trigrams = get_top_ngrams(df, n=3, top_k=20)

print('Top 20 Trigrams:')
for ngram, count in top_trigrams:
    print(f'  {ngram:>40}: {count:,}')

In [ ]:
fig = plot_top_ngrams(top_trigrams, title='Top 20 Trigrams')
fig.savefig(os.path.join(FIGURES_DIR, 'top_trigrams.png'), bbox_inches='tight')
plt.show()

---
## 8. Vocabulary Analysis

In [ ]:
vocab_stats = get_vocabulary_stats(df)

print(f"Unique Words (Vocabulary Size): {vocab_stats['unique_words']:,}")
print(f"Total Words:                    {vocab_stats['total_words']:,}")

---
## 9. Data Quality Assessment

In [ ]:
quality_issues = assess_data_quality(df)

print('Data Quality Report:')
for issue, count in quality_issues.items():
    status = '✓ OK' if count == 0 else f'⚠ {count:,} issues'
    print(f'  {issue:>25}: {status}')

---
## 10. Generate Reports

In [ ]:
# Generate EDA Summary Report (Markdown)
report_content = generate_eda_report(
    overview=overview,
    label_dist=label_dist,
    text_stats=text_stats,
    vocab_stats=vocab_stats,
    imbalance=imbalance,
    duplicate_info=duplicate_info,
    missing_df=missing_df,
    quality_issues=quality_issues,
)
save_report(report_content, os.path.join(REPORTS_DIR, 'eda_summary.md'))

# Export CSV reports
export_dataset_overview(overview, os.path.join(REPORTS_DIR, 'dataset_overview.csv'))
export_label_distribution(label_dist, os.path.join(REPORTS_DIR, 'label_distribution.csv'))
export_text_statistics(text_stats, os.path.join(REPORTS_DIR, 'text_statistics.csv'))
export_vocabulary_statistics(vocab_stats, os.path.join(REPORTS_DIR, 'vocabulary_statistics.csv'))

print('\n✓ All reports generated successfully.')

---
## Summary

EDA is complete. All reports and visualizations have been saved to:

- `reports/eda/eda_summary.md` — Full EDA report
- `reports/eda/dataset_overview.csv`
- `reports/eda/label_distribution.csv`
- `reports/eda/text_statistics.csv`
- `reports/eda/vocabulary_statistics.csv`
- `reports/eda/figures/` — All visualizations

**Next Step**: `02_preprocessing.ipynb` — Text Cleaning & Preprocessing